# RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 설치
- `pip install ragas rapidfuzz`

## RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 주요 평가지표
### Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# RAGAS 평가 실습

In [ ]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [ ]:
# docker run -p 6333:6333 -p 6334:6334   -v qdrant_storage:/qdrant/storage   qdrant/qdrant

In [4]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_qdrant import FastEmbedSparse, QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, SparseVectorParams, VectorParams
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv

load_dotenv()


True

In [27]:
# ##############################################################
# 데이터 준비
##############################################################

def load_and_split_olympic_data(file_path="data/olympic_wiki.md"):
    with open(file_path, "r", encoding="utf-8") as fr:
        olympic_text = fr.read()

    # Split
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "H1"),
            ("##", "H2"),
            ("###", "H3"),
        ],
    )

    return splitter.split_text(olympic_text)

In [28]:
#################################################################
# Vector DB 연결
# retriever 생성
#################################################################

def get_vectorstore(collection_name: str = "olympic_info_wiki"):


    dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

    client = QdrantClient(url="http://localhost:6333")

    # 컬렉션 삭제
    if client.collection_exists(collection_name):
        result = client.delete_collection(collection_name=collection_name)

    # 컬렉션 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE),
      
    )

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,    
        embedding=dense_embeddings
    )
    
    ######################################
    # Document들 추가
    ######################################
    documents = load_and_split_olympic_data()
    vectorstore.add_documents(documents=documents)

    return vectorstore


def get_retriever(vectorstore, k: int = 5):
    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )
    return retriever

In [29]:
vectorstore = get_vectorstore()

retriever = get_retriever(vectorstore)
retriever

VectorStoreRetriever(tags=['QdrantVectorStore', 'OpenAIEmbeddings'], vectorstore=<langchain_qdrant.qdrant.QdrantVectorStore object at 0x000001F3AEEA6E40>, search_kwargs={'k': 5})

In [30]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from operator import itemgetter

vectorstore = get_vectorstore()
retriever = get_retriever(vectorstore)

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(
    template=prompt_txt
)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

def format_doc_to_str(documents:list[Document])->list[str]:
    """
    VectorStore에 조회한 문서들(list[Document])에서 내용(page_content)만 추출해서 list[str] 로 반환.
    RAGAS 평가시 context는 각 검색한 문서를 list[str] 로 받기 때문에 이렇게 처리.
    
    Args:
        documents(list[Document]): [Document(..), Document(...), ..]}
    Returns:
        list[str]: 각 문서의 내용만 추출해서 리스트에 담는다.
    """
    return [doc.page_content for doc in documents]

# RAG 체인 -> 평가데이터셋을 만드는 RAG Chain
#          -> 최종 응답: LLM의 응답(str), 검색한 문서들(list[str])
chain = RunnablePassthrough() | {
    "context":retriever | format_doc_to_str,
    "query":RunnablePassthrough()
} | { 
    "response": prompt | model | parser,
    "retrieved_context": itemgetter("context")
}
# RunnablePassthrough() -> LCEL 체인을 만들려면 구성요소중 하나가 Runnable이어야 하는데
# 이 체인은 dict | dict 구조기 때문에 앞에 추가.

In [6]:
res = chain.invoke("1회 올림픽은 언제 어디서 열렸지")

In [7]:
print(res.keys())

dict_keys(['response', 'retrieved_context'])


In [8]:
res['response']

'기원전 776년에 그리스 올림피아에서 열렸습니다.'

In [9]:
res['retrieved_context']

['고대의 올림픽 경기(올림피아 경기)는 고대 그리스의 여러 도시 국가의 대표선수들이 모여 벌인 일련의 시합이었으며, 육상 경기가 주 종목이지만 격투기와 전차 경기도 열렸다. 그리고 패배하면 죽기도 하였다. 고대 올림픽의 유래는 수수께끼로 남아있다. 잘 알려진 신화로는 헤라클레스와 그의 아버지인 제우스가 올림픽의 창시자였다는 것이다. 전설에 따르면 이 경기를 최초로 \'올림픽\'이라고 부르고, 4년마다 대회를 개최하는 관례를 만든 사람이 헤라클레스라고 한다. 어떤 전설에서는 헤라클레스가 이른바 헤라클레스의 12업을 달성한 뒤에 제우스를 기리고자 올림픽 경기장을 지었다고 한다. 경기장이 완성되자 헤라클레스는 일직선으로 200 걸음을 걸었으며, 이 거리를 "스타디온"이라 불렀는데, 후에 이것이 길이 단위인 \'스타디온\'(그리스어: στάδιον → 라틴어: 영어: stadium)이 되었다. 또 다른 설로는 \'올림픽 휴전\'(그리스어: ἐκεχειρία 에케케이리아[*])이라는 고대 그리스의 관념이 최초의 올림피아 경기와 관련이 있다고 한다. \'올림픽 휴전\'이란 어느 도시 국가라도 올림피아 경기 기간 중에 다른 나라를 침범하면 그에 대한 응징을 받을 수 있다는 뜻으로, "올림픽 기간에는 전쟁하지 말 것"으로 요약할 수 있다.  \n고대 올림피아 경기가 처음 열린 시점은 보통 기원전 776년으로 인정되고 있는데, 이 연대는 그리스 올림피아에서 발견된 비문에 근거를 둔 것이다. 이 비문의 내용은 달리기 경주 승자 목록이며 기원전 776년부터 4년 이후 올림피아 경기 마다의 기록이 남겨져 있다. 고대 올림픽의 종목으로는 육상, 5종 경기(원반던지기, 창던지기, 달리기, 레슬링, 멀리뛰기), 복싱, 레슬링, 승마 경기가 있었다. 전설에 따르면 엘리스의 코로이보스가 최초로 올림피아 경기에서 우승한 사람이라고 한다.  \n고대 올림피아 경기는 근본적으로 종교적인 중요성을 띄고 있었는데, 스포츠 경기를 할 때는 제우스(올림피아의 제우스 신전에는 페이디아스가 만든 제우스 

# RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>           from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [1]:
# 주피터노트북 환경에서 비동기적 처리 위해
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [2]:
from ragas.testset import TestsetGenerator

In [ ]:
##################################################################################################
# testset -> Context들(문서들) - [질문 - 정답답변 + (Retriever가 찾은 문서 + LLM 정답: Chain 생성)]
# 1. Context(문서들)을 추출 -> TestsetGenerator -> 질문과 정답답변 생성.

import random
# 데이터셋을 생성할 때 사용할 Context를 추출.
client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "olympic_info_wiki"

# 전체 저장된 문서 중에서 K개만 sampling
info = client.get_collection(COLLECTION_NAME)
total_docs = info.points_count # 총 문서개수 조회

results, _ = client.scroll(
    collection_name= COLLECTION_NAME,
    limit = total_docs,
)
# 랜덤하게 K(5)개를 sampling
sample_docs:"list[PointStruct]" = random.sample(results, 5) # 리스트에서 랜덤하게 k(5)개를 추출 (k: 데이터셋 개수)

# PointStruct - payload: page_content, metadata
# page_content만 추출해서 list[str]
docs = [point.payload['page_content'] for point in sample_docs]

In [8]:
docs

["1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장으로 있던 1952년부터 1972년까지, 에이버리 브런디지는 그 어떤 상업적인 관심도 올림픽과 연계를 꾀하는 것을 거부했다. 협력 스폰서의 관심이 IOC의 결정에 지나치게 간섭할 우려가 있다고 생각했기 때문이다. 브런디지가 이러한 수익창출을 거부했다는 것은 IOC가 스폰서 계약이나 올림픽 상징의 사용에 대한 협상을 포기했다는 뜻이다. 브런디지가 IOC에 200만 달러를 남기고 은퇴한 후 8년 뒤에 IOC의 잔고는 4500만 달러로 늘어났다. 그 이유는 IOC가 처음으로 텔레비전 중계권을 판매하고 스폰서와 계약함으로써 올림픽의 팽창을 노리는 이데올로기(관념)의 변화가 있었기 때문이다. 1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 IOC의 재정적 독립이었을 정도로 상업성에 대한 시각도 많이 달라졌다.  \n1984년 하계 올림픽은 올림픽 역사에 있어서 획기적인 순간이었다. 피터 워버로스(Peter Ueberroth)의 지휘하에 있던 LA 하계 올림픽 조직위원회는 그 당시 2억 2500만 달러라는 전례가 없던 이익을 얻었다. 왜냐하면 조직위원회는 독점스폰서에 대한 권리를 판매하였고 그로 인해 이익을 창출할 수 있었기 때문이다. IOC는 이러한 재정적 후원 권리를 통제하기 위한 방법을 강구했다. 1년 뒤인 1985년 사마란치 IOC 위원장은 올림픽 브랜드를 만들어내기 위한 '올림픽 프로그램(The Olympic Program, TOP)'을 설립했다. TOP의 회원들은 '독점'적이고, '비싼 비용'을 IOC에 지불해야만 한다. 4년마다 TOP 회원들은 5000만 달러를 내야 한다. 그 대가로 TOP의 회원은 생산하는 물품에 올림픽 상징인 오륜기를 국제적, 독점적으로 사용하고 출판물이나 광고에도 이용할 수 있는 혜택을 누리게 된다.",
 "올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다.  \n#### 개막식  \n개막식 때는 

In [10]:
total_docs
sample_docs

[Record(id='b87e4596-2048-4d85-aacd-8be8beb5ca14', payload={'page_content': "1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장으로 있던 1952년부터 1972년까지, 에이버리 브런디지는 그 어떤 상업적인 관심도 올림픽과 연계를 꾀하는 것을 거부했다. 협력 스폰서의 관심이 IOC의 결정에 지나치게 간섭할 우려가 있다고 생각했기 때문이다. 브런디지가 이러한 수익창출을 거부했다는 것은 IOC가 스폰서 계약이나 올림픽 상징의 사용에 대한 협상을 포기했다는 뜻이다. 브런디지가 IOC에 200만 달러를 남기고 은퇴한 후 8년 뒤에 IOC의 잔고는 4500만 달러로 늘어났다. 그 이유는 IOC가 처음으로 텔레비전 중계권을 판매하고 스폰서와 계약함으로써 올림픽의 팽창을 노리는 이데올로기(관념)의 변화가 있었기 때문이다. 1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 IOC의 재정적 독립이었을 정도로 상업성에 대한 시각도 많이 달라졌다.  \n1984년 하계 올림픽은 올림픽 역사에 있어서 획기적인 순간이었다. 피터 워버로스(Peter Ueberroth)의 지휘하에 있던 LA 하계 올림픽 조직위원회는 그 당시 2억 2500만 달러라는 전례가 없던 이익을 얻었다. 왜냐하면 조직위원회는 독점스폰서에 대한 권리를 판매하였고 그로 인해 이익을 창출할 수 있었기 때문이다. IOC는 이러한 재정적 후원 권리를 통제하기 위한 방법을 강구했다. 1년 뒤인 1985년 사마란치 IOC 위원장은 올림픽 브랜드를 만들어내기 위한 '올림픽 프로그램(The Olympic Program, TOP)'을 설립했다. TOP의 회원들은 '독점'적이고, '비싼 비용'을 IOC에 지불해야만 한다. 4년마다 TOP 회원들은 5000만 달러를 내야 한다. 그 대가로 TOP의 회원은 생산하는 물품에 올림픽 상징인 오륜기를 국제적, 독점적으로 사용하고 출판물이나 광고에도 이용할 수 있는 혜택을 

In [11]:
# 테스트 셋을 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import OpenAI, OpenAIEmbeddings
# TestsetGenerator는 gpt-5 이후 버전은 사용할 수 없다. (호환성 문제 / Temp)
## Langchain의 LLM 모델과 Embedding 모델 -> RAGAS에서 사용할 수 있도록 변환 (Wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

generator = TestsetGenerator(
    llm= generator_llm,
    embedding_model = generator_embeddings,
    llm_context="""
- 사람들이 올림픽에 대해서 궁금해 할 만한 질문들을 생성한다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법을 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON 문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
""" # 질문/답변을 생성할 때 LLM에게 전달할 System Prompt를 설정.
)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_23384\3545982627.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23384\3545982627.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [12]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용, 테스트셋 개수 (질문-답변 개수)
)

Applying SummaryExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

Node 0a113584-e02f-4398-b028-e990e72926da does not have a summary. Skipping filtering.
Node 960706e9-1e28-4949-98b6-b7f199ee5661 does not have a summary. Skipping filtering.
Node c049f95f-522f-4bca-9713-bfcddfe091b0 does not have a summary. Skipping filtering.
Node 44afef2c-6c11-4d34-90bf-4203c0dd20cc does not have a summary. Skipping filtering.
Node 72c731f5-9bb2-45df-9c78-a4b272666444 does not have a summary. Skipping filtering.


Applying EmbeddingExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [13]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='후안 안토니오 사마란치가 IOC 위원장으로 당선된 후 올림픽 상업성에 어떤 변화가 있었나요?', retrieved_contexts=None, reference_contexts=["1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장으로 있던 1952년부터 1972년까지, 에이버리 브런디지는 그 어떤 상업적인 관심도 올림픽과 연계를 꾀하는 것을 거부했다. 협력 스폰서의 관심이 IOC의 결정에 지나치게 간섭할 우려가 있다고 생각했기 때문이다. 브런디지가 이러한 수익창출을 거부했다는 것은 IOC가 스폰서 계약이나 올림픽 상징의 사용에 대한 협상을 포기했다는 뜻이다. 브런디지가 IOC에 200만 달러를 남기고 은퇴한 후 8년 뒤에 IOC의 잔고는 4500만 달러로 늘어났다. 그 이유는 IOC가 처음으로 텔레비전 중계권을 판매하고 스폰서와 계약함으로써 올림픽의 팽창을 노리는 이데올로기(관념)의 변화가 있었기 때문이다. 1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 IOC의 재정적 독립이었을 정도로 상업성에 대한 시각도 많이 달라졌다.  \n1984년 하계 올림픽은 올림픽 역사에 있어서 획기적인 순간이었다. 피터 워버로스(Peter Ueberroth)의 지휘하에 있던 LA 하계 올림픽 조직위원회는 그 당시 2억 2500만 달러라는 전례가 없던 이익을 얻었다. 왜냐하면 조직위원회는 독점스폰서에 대한 권리를 판매하였고 그로 인해 이익을 창출할 수 있었기 때문이다. IOC는 이러한 재정적 후원 권리를 통제하기 위한 방법을 강구했다. 1년 뒤인 1985년 사마란치 IOC 위원장은 올림픽 브랜드를 만들어내기 위한 '올림픽 프로그램(The Olympic Program, TOP)'을 설립했다. TOP의 회원들은 '독점'적이고, '비싼 비용'을 IOC에 지불해야만 한다. 4년마다 TOP 회원들은

In [20]:
sample1 = testset.samples[9].eval_sample # 10개중 첫번째 테스트 데이터
print("사용자 질문:", sample1.user_input)
print("Context:", sample1.reference_contexts) # 질문과 답변을 만들 때 사용한 context (검색시 찾아야 하는 문서)
print("생성된 답변(정답):", sample1.reference)

##### 평가 대상 RAG System을 이용해서 채워 넣어야 한다.
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)

사용자 질문: 올림픽 개막식과 폐막식에서 그리스가 어떤 특별한 역할을 하며, 이러한 전통이 올림픽의 상징과 어떻게 연결되어 있는지 설명하시오.
Context: ['<1-hop>\n\n올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용한다. 오륜기로 잘 알려져 있는 올림픽기는 5개의 둥근 고리가 얽혀있으며 각 원마다 5대륙을 상징한다.(남아메리카와 북아메리카는 아메리카로 합쳐 있다.) 파랑, 노랑, 검정, 초록, 빨간 고리에 흰색 바탕은 올림픽 기를 나타낸다. 이 색들이 선택된 이유는 모든 국기에서 적어도 이 5개의 색 중 하나를 가지고 있기 때문이다. 올림픽 기가 채택된 것은 1914년이지만 1920년 하계 올림픽 때부터 사용되기 시작했다. 이 때부터 올림픽 기는 올림픽을 기념하기 위해 올림픽이 열리는 기간 동안 게양한다.  \n올림픽 표어는 라틴어로 Citius, Altius, Fortius이며 "더 빨리, 더 높게, 더 힘차게"라는 뜻이다. 쿠베르탱의 이상은 올림픽 선서에 더 잘 나타나 있다.  \n- 올림픽 선서 중에서\n- "인생에서 가장 중요한 것은 승리가 아니라 이를 위해 분투하는 것이고, 올림픽에서 가장 중요한 것 역시 승리가 아니라 참가 자체에 의의가 있다. 우리에게 있어 본질은 정복하는 것이 아니라 잘 싸우는 것이다."  \n매 올림픽이 시작되기 몇 개월전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아에서 올림픽 성화가 채화된다. 여자 배우가 마치 여자 사제인 것처럼 연기해서 태양광선을 포물면 거울(오목 거울의 하나)의 안쪽에 집중시켜서 점화한다. 그 후에 여자는 첫 번째 성화 봉송 주자에게 성화를 넘기고 개최도시의 개막식이 열리는 올림픽 경기장까지 여러 사람의 손을 거쳐서 올림픽 성화는 전달된다. 올림픽 상징으로서의 올림픽 성화는 1928년 하계 올림픽 때 이미 있었지만, 성화 봉송은 1936년 하계 올림픽때 독일 정부의 나치즘 선전의 일환으로 처음 시행된 것이 그 유래이다.  \n개최도시의 문화적 상징물을 

In [21]:
# 생성된 Testset을 Pandas DataFrame으로 변환

eval_df = testset.to_pandas()
eval_df.shape

(10, 7)

In [22]:
eval_df.head(3)

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,후안 안토니오 사마란치가 IOC 위원장으로 당선된 후 올림픽 상업성에 어떤 변화가 ...,[1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장...,"1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 ...",Sports Business Analyst,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
1,아테네 올림픽 개막식에서 그리스 선수단의 입장 순서는 어떻게 되었나요?,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하...,Olympics Enthusiast,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer
2,"동계 올림픽 뭐뭐 있나, 몇 개 부문 있고 언제부터 어떤 종목 계속 있었는지, 그리...",[올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있...,"동계 올림픽은 7개 부문으로 이루어져 있고, 크로스컨트리, 피겨 스케이팅, 아이스 ...",Olympics Enthusiast,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer


In [31]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']
resp = chain.invoke(q) # dict[response, retrieved_context]

In [33]:
resp['response']

'후안 안토니오 사마란치가 IOC 위원장으로 당선된 1980년대부터 올림픽은 **상업성을 받아들이는 방향으로 변화**하기 시작했습니다.  \n구체적으로는 **국제적인 스폰서를 맞아들여 올림픽 관련 상품과 올림픽 브랜드를 연계**시키는 방식으로 바뀌었고, 1985년에는 **‘올림픽 프로그램(TOP)’**을 설립해 올림픽 상징의 독점적 사용과 스폰서 계약을 본격화했습니다.'

In [32]:
resp['retrieved_context']

['처음에 IOC는 스폰서에게서 자금제공을 받는 것을 거부했었다. 이런 방침은 에이버리 브런디지가 IOC 위원장이었던 1972년까지 유지되어 왔었다. 이 당시 IOC는 텔레비전 같은 미디어들이 갖는 잠재성과 큰 수익을 가져오는 광고시장에 대해 조사했었다. 그 후 1980년대 후안 안토니오 사마란치 위원장 시절부터 올림픽은 올림픽 관련 상품과 올림픽 브랜드를 연계시키려는 국제적인 스폰서를 맞아들임으로써 변화가 시작되었다.',
 "1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장으로 있던 1952년부터 1972년까지, 에이버리 브런디지는 그 어떤 상업적인 관심도 올림픽과 연계를 꾀하는 것을 거부했다. 협력 스폰서의 관심이 IOC의 결정에 지나치게 간섭할 우려가 있다고 생각했기 때문이다. 브런디지가 이러한 수익창출을 거부했다는 것은 IOC가 스폰서 계약이나 올림픽 상징의 사용에 대한 협상을 포기했다는 뜻이다. 브런디지가 IOC에 200만 달러를 남기고 은퇴한 후 8년 뒤에 IOC의 잔고는 4500만 달러로 늘어났다. 그 이유는 IOC가 처음으로 텔레비전 중계권을 판매하고 스폰서와 계약함으로써 올림픽의 팽창을 노리는 이데올로기(관념)의 변화가 있었기 때문이다. 1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 IOC의 재정적 독립이었을 정도로 상업성에 대한 시각도 많이 달라졌다.  \n1984년 하계 올림픽은 올림픽 역사에 있어서 획기적인 순간이었다. 피터 워버로스(Peter Ueberroth)의 지휘하에 있던 LA 하계 올림픽 조직위원회는 그 당시 2억 2500만 달러라는 전례가 없던 이익을 얻었다. 왜냐하면 조직위원회는 독점스폰서에 대한 권리를 판매하였고 그로 인해 이익을 창출할 수 있었기 때문이다. IOC는 이러한 재정적 후원 권리를 통제하기 위한 방법을 강구했다. 1년 뒤인 1985년 사마란치 IOC 위원장은 올림픽 브랜드를 만들어내기 위한 '올림픽 프로그램(The Olympic Program, 

In [36]:
# testset의 모든 데이터에 대한 llm 응답과 retriever의 검색 결과를 추가.
response_list = [] # LLM 응답들을 저장할 리스트
retrieved_context_list = [] # retriever가 검색한 문서들을 저장할 리스트

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp['response'])
    retrieved_context_list.append(resp['retrieved_context'])

In [38]:
print(len(response_list), len(retrieved_context_list))

10 10


In [40]:
response_list

['후안 안토니오 사마란치가 IOC 위원장으로 당선된 뒤, 올림픽은 **상업성을 받아들이는 방향으로 변화**했습니다.  \n구체적으로는 **국제적 스폰서를 받아들이고**, **올림픽 관련 상품과 올림픽 브랜드를 연계**시키는 움직임이 시작되었습니다. 또한 **텔레비전 중계권 판매와 스폰서 계약**을 통해 재정적 독립과 수익 창출을 추구하게 되었습니다.',
 '정보가 부족해 답을 할 수없습니다.',
 '동계 올림픽에 대한 내용은 context에 다음과 같이 있습니다.\n\n- **동계 올림픽은 7개 부문**으로 이루어져 있습니다.\n- **1924년 동계 올림픽부터 빠짐없이 정식종목이었던 종목**은 다음 6개입니다:\n  - 크로스컨트리\n  - 피겨 스케이팅\n  - 아이스 하키\n  - 노르딕 복합\n  - 스키 점프\n  - 스피드 스케이팅\n\n- **동계 올림픽 종목이 정해지는 기준**에 대해서는, context에는 동계 종목에 대한 별도 선정 기준은 직접적으로 적혀 있지 않고, 올림픽 종목 전반에 대한 기준만 나와 있습니다.  \n  IOC 산하 **올림픽 프로그램 위원회**는 올림픽 종목 포함 여부를 판단할 때 다음 **7가지 기준**을 제시합니다:\n  1. 역사\n  2. 전통\n  3. 보편성\n  4. 인기도와 잠재성\n  5. 선수의 건강\n  6. 연맹의 스포츠 관리 능력\n  7. 스포츠를 여는 데 필요한 비용\n\n- **IOC가 정식종목을 정하는 방식**은 다음과 같습니다:\n  - 올림픽 끝난 뒤 처음 열리는 **IOC 총회** 때마다 정식종목 신청이 가능함\n  - **IOC 위원들의 투표**로 결정됨\n  - **재적 위원 수의 과반수 이상 찬성표**를 얻어야 정식종목으로 인정됨\n\n즉, context 기준으로 보면 동계 올림픽의 기본 정보는 위와 같고, 종목 채택 방식은 IOC 총회 투표와 과반수 찬성입니다.',
 '올림픽 성화는 **매 올림픽이 시작되기 몇 개월 전, 고대 그리스의 올림피아에서 채화**됩니다. 이는 **고대

In [41]:
retrieved_context_list

[['처음에 IOC는 스폰서에게서 자금제공을 받는 것을 거부했었다. 이런 방침은 에이버리 브런디지가 IOC 위원장이었던 1972년까지 유지되어 왔었다. 이 당시 IOC는 텔레비전 같은 미디어들이 갖는 잠재성과 큰 수익을 가져오는 광고시장에 대해 조사했었다. 그 후 1980년대 후안 안토니오 사마란치 위원장 시절부터 올림픽은 올림픽 관련 상품과 올림픽 브랜드를 연계시키려는 국제적인 스폰서를 맞아들임으로써 변화가 시작되었다.',
  "1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장으로 있던 1952년부터 1972년까지, 에이버리 브런디지는 그 어떤 상업적인 관심도 올림픽과 연계를 꾀하는 것을 거부했다. 협력 스폰서의 관심이 IOC의 결정에 지나치게 간섭할 우려가 있다고 생각했기 때문이다. 브런디지가 이러한 수익창출을 거부했다는 것은 IOC가 스폰서 계약이나 올림픽 상징의 사용에 대한 협상을 포기했다는 뜻이다. 브런디지가 IOC에 200만 달러를 남기고 은퇴한 후 8년 뒤에 IOC의 잔고는 4500만 달러로 늘어났다. 그 이유는 IOC가 처음으로 텔레비전 중계권을 판매하고 스폰서와 계약함으로써 올림픽의 팽창을 노리는 이데올로기(관념)의 변화가 있었기 때문이다. 1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 IOC의 재정적 독립이었을 정도로 상업성에 대한 시각도 많이 달라졌다.  \n1984년 하계 올림픽은 올림픽 역사에 있어서 획기적인 순간이었다. 피터 워버로스(Peter Ueberroth)의 지휘하에 있던 LA 하계 올림픽 조직위원회는 그 당시 2억 2500만 달러라는 전례가 없던 이익을 얻었다. 왜냐하면 조직위원회는 독점스폰서에 대한 권리를 판매하였고 그로 인해 이익을 창출할 수 있었기 때문이다. IOC는 이러한 재정적 후원 권리를 통제하기 위한 방법을 강구했다. 1년 뒤인 1985년 사마란치 IOC 위원장은 올림픽 브랜드를 만들어내기 위한 '올림픽 프로그램(The Olympic Program

In [45]:
###################################################
# eval_df에 컬럼으로 추가
###################################################
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,response,retrieved_context,retrieved_contexts
0,후안 안토니오 사마란치가 IOC 위원장으로 당선된 후 올림픽 상업성에 어떤 변화가 ...,[1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장...,"1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 ...",Sports Business Analyst,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,"후안 안토니오 사마란치가 IOC 위원장으로 당선된 뒤, 올림픽은 **상업성을 받아들...",[처음에 IOC는 스폰서에게서 자금제공을 받는 것을 거부했었다. 이런 방침은 에이버...,[처음에 IOC는 스폰서에게서 자금제공을 받는 것을 거부했었다. 이런 방침은 에이버...
1,아테네 올림픽 개막식에서 그리스 선수단의 입장 순서는 어떻게 되었나요?,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하...,Olympics Enthusiast,MISSPELLED,MEDIUM,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...","[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#..."
2,"동계 올림픽 뭐뭐 있나, 몇 개 부문 있고 언제부터 어떤 종목 계속 있었는지, 그리...",[올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있...,"동계 올림픽은 7개 부문으로 이루어져 있고, 크로스컨트리, 피겨 스케이팅, 아이스 ...",Olympics Enthusiast,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,동계 올림픽에 대한 내용은 context에 다음과 같이 있습니다.\n\n- **동계...,[올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있...,[올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있...
3,"올림픽 성화와 그리스의 관계는 무엇이며, 성화가 어떻게 채화되는지 설명해 주세요.",[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,매 올림픽이 시작되기 몇 개월 전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아...,Sports Business Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer,"올림픽 성화는 **매 올림픽이 시작되기 몇 개월 전, 고대 그리스의 올림피아에서 채...",[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...
4,"1948년 런던 올림픽 그때 뭐 있었고 왜 중요한지 설명해줘, 잘 모르겠는데 그때 ...",[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,1948년에 루드비히 구트만 경은 제2차 세계대전에 참전한 군인들의 사회 복귀를 위...,Olympic Sports Program Coordinator,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,정보가 부족해 답을 할 수없습니다.,"[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...","[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실..."


In [46]:
# eval_df를 RAGAS의 평가 데이터셋 타입으로 변환.
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=10)

In [48]:
#############################
# 평가
#############################
from ragas.metrics import (
    LLMContextRecall, # Context Recall
    LLMContextPrecisionWithReference, # Context Precision
    Faithfulness,
    AnswerRelevancy
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_23384\1150867979.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23384\1150867979.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23384\1150867979.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\User

In [51]:
# 평가할 때 사용할 LLM, Embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

# Metric(평가지표) 객체를 List로 묶어준다.
## 내가 평가할 지표들만 묶어준다.
metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings= eval_embeddings)
]
# 평가진행
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_23384\4169728312.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_23384\4169728312.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Playdata\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001F36A7BE800> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Playdata\AppData\Roaming\uv\python\cpython-3.13-windows-x86_64-none\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x000001F36A7BE800> is already entered
Task was destroyed but it is pending!
task: 

In [52]:
eval_result

{'context_recall': 1.0000, 'llm_context_precision_with_reference': 0.8333, 'faithfulness': 0.6667, 'answer_relevancy': 0.4421}

In [ ]:
# print(type(eval_result))
## 개별 평가데이터에 대한 평가점수.
result_df = eval_result.to_pandas()

In [54]:
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,후안 안토니오 사마란치가 IOC 위원장으로 당선된 후 올림픽 상업성에 어떤 변화가 ...,[처음에 IOC는 스폰서에게서 자금제공을 받는 것을 거부했었다. 이런 방침은 에이버...,"후안 안토니오 사마란치가 IOC 위원장으로 당선된 뒤, 올림픽은 **상업성을 받아들...","1980년에 IOC위원장으로 후안 안토니오 사마란치가 당선되었을 때, 그의 소원이 ...",1.0,1.000000,NaN,0.925351
1,아테네 올림픽 개막식에서 그리스 선수단의 입장 순서는 어떻게 되었나요?,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",정보가 부족해 답을 할 수없습니다.,그리스 아테네에서 열린 2004년 하계 올림픽에서는 그리스 국기가 맨 처음에 입장하...,1.0,1.000000,NaN,0.000000
2,"동계 올림픽 뭐뭐 있나, 몇 개 부문 있고 언제부터 어떤 종목 계속 있었는지, 그리...",[올림픽 경기 종목은 총 33개부문 52개 종목에서 약 400개의 경기로 이루어져있...,동계 올림픽에 대한 내용은 context에 다음과 같이 있습니다.\n\n- **동계...,"동계 올림픽은 7개 부문으로 이루어져 있고, 크로스컨트리, 피겨 스케이팅, 아이스 ...",1.0,1.000000,NaN,0.665328
3,"올림픽 성화와 그리스의 관계는 무엇이며, 성화가 어떻게 채화되는지 설명해 주세요.",[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,"올림픽 성화는 **매 올림픽이 시작되기 몇 개월 전, 고대 그리스의 올림피아에서 채...",매 올림픽이 시작되기 몇 개월 전에 고대 그리스에서 제사를 지냈던 그리스의 올림피아...,1.0,1.000000,NaN,0.678446
4,"1948년 런던 올림픽 그때 뭐 있었고 왜 중요한지 설명해줘, 잘 모르겠는데 그때 ...","[쿠베르탱의 생각과는 달리, 올림픽이 세계에 완벽한 평화를 가져다주지는 못했다. 실...",정보가 부족해 답을 할 수없습니다.,1948년에 루드비히 구트만 경은 제2차 세계대전에 참전한 군인들의 사회 복귀를 위...,NaN,0.000000,NaN,0.000000
5,"올림픽 상징인 오륜기가 상업적으로 어떻게 활용되었으며, 오륜기의 의미는 무엇인가?",[올림픽에서는 올림픽 헌장에 구체적으로 나타난 이상이나 철학을 표현하는 상징을 사용...,오륜기는 상업적으로는 IOC가 **올림픽 브랜드와 함께 독점적으로 사용·판매되는 상...,올림픽 상징인 오륜기는 1985년 IOC가 설립한 올림픽 프로그램(TOP)을 통해 ...,NaN,0.500000,NaN,0.531642
6,1920년 하계 올림픽 개막식이랑 폐막식에서 시작된 전통은 뭐에요?,"[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",정보가 부족해 답을 할 수없습니다.,"1920년 하계 올림픽 개막식에서는 개막식의 기본 토대가 만들어졌고, 폐막식에서는 ...",NaN,1.000000,NaN,0.000000
7,"1984년 하계 올림픽이 올림픽의 상업화와 재정 구조에 어떤 변화를 가져왔으며, 이...",[1950년대까지 IOC는 적은 예산으로 운영되어 왔다. 에이버리 브런디지가 위원장...,1984년 하계 올림픽은 올림픽의 **상업화와 재정 구조가 크게 바뀐 획기적인 전환...,1984년 하계 올림픽은 LA 하계 올림픽 조직위원회가 독점스폰서 권리를 판매하여 ...,1.0,0.833333,1.0,0.900933
8,1960년 하계 올림픽과 관련하여 패럴림픽의 시작과 그 이후 올림픽 개최 도시와의 ...,[패럴림픽(Paralympic)은 신체·감각 장애가 있는운동 선수가 참가하는 국제 ...,"1960년 로마에서 열린 하계 올림픽 때, 루드비히 구트만이 400명의 선수들을 “...",패럴림픽은 1960년 로마에서 열린 하계 올림픽 때 루드비히 구트만이 400명의 선...,1.0,1.000000,1.0,0.719570
9,"올림픽 개막식과 폐막식에서 그리스가 어떤 특별한 역할을 하며, 이러한 전통이 올림픽...","[올림픽에서 이루어지는 주요 행사로는 개막식, 폐막식, 시상식 등이 있다. \n#...",정보가 부족해 답을 할 수없습니다.,올림픽 개막식에서는 올림픽의 발상지라는 영예를 가진 그리스가 전통적으로 맨 처음에 ...,1.0,1.000000,0.0,0.000000
